<a href="https://colab.research.google.com/github/Guliko24/macmillan-research/blob/main/macmillan_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 1: Setup & Environment

In [ ]:
# @title 📦 Install Dependencies & Setup Output Folder
!pip install requests beautifulsoup4 pandas lxml matplotlib seaborn wordcloud -q

import os
from google.colab import drive

# Mount Google Drive for persistent storage
drive.mount('/content/drive', force_remount=True)
OUTPUT_DIR = '/content/drive/MyDrive/MacmillanResearch'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Environment ready. Outputs will save to: {OUTPUT_DIR}")

Mounted at /content/drive
✅ Environment ready. Outputs will save to: /content/drive/MyDrive/MacmillanResearch


# Cell 2: DOM Inspector (Run First to Verify Selectors)

In [ ]:
# @title 🔍 Step 0: Inspect Forum DOM Structure (Run Once)
import requests
from bs4 import BeautifulSoup

url = "https://community.macmillan.org.uk/cancer_types/breast-cancer-forum"
headers = {
    "User-Agent": "Mozilla/5.0 (compatible; CancerResearchBot/1.0; non-commercial health research)"
}

resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

# Print sample structure to verify selectors
print("📌 Checking thread cards...")
cards = soup.select("li[data-qa='discussion-card'], .DiscussionItem, .forum-discussions li")
if cards:
    print(f"✅ Found {len(cards)} thread cards. Sample structure:\n{cards[0].prettify()[:500]}...")
else:
    print("⚠️ No cards found with default selectors. Check page source and update selectors below.")
    print("🔍 Tip: Right-click a thread title > Inspect > copy selector")

📌 Checking thread cards...
⚠️ No cards found with default selectors. Check page source and update selectors below.
🔍 Tip: Right-click a thread title > Inspect > copy selector


# Cell 3: Scraper **Engine**

In [ ]:
# @title 🕷️ Step 1: Macmillan Breast Cancer Forum Scraper
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import logging
import re
from datetime import datetime
from urllib.parse import urljoin

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; CancerResearchBot/1.0; non-commercial health research)"
}
MAX_PAGES = 8  # ~80-150 posts (adjust as needed)
DELAY = 3.5    # Polite delay
BASE_URL = "https://community.macmillan.org.uk/cancer_types/breast-cancer-forum"

def get_page_url(page):
    if page == 1: return BASE_URL
    return f"{BASE_URL}?pifragment-15992={page}"

def extract_threads(soup, base_url):
    threads = []
    # Fallback selectors: adjust if DOM changes
    for card in soup.select("li[data-qa='discussion-card'], .DiscussionItem, .forum-discussions li"):
        try:
            title_el = card.select_one("h3 a, .DiscussionTitle a, a.link--title")
            if not title_el: continue

            title = title_el.get_text(strip=True)
            post_url = urljoin(base_url, title_el.get("href", ""))

            meta = card.select_one(".DiscussionMeta, .stats, .meta")
            replies = 0
            views = 0
            if meta:
                r = re.search(r'(\d+)\s+repl', meta.get_text())
                v = re.search(r'(\d+)\s+view', meta.get_text())
                replies = int(r.group(1)) if r else 0
                views = int(v.group(1)) if v else 0

            threads.append({
                "title": title,
                "url": post_url,
                "replies": replies,
                "views": views,
                "scraped_at": datetime.now().isoformat()
            })
        except Exception as e:
            logging.warning(f"Parse error: {e}")
    return threads

def scrape_breast_cancer_forum():
    all_threads = []
    for page in range(1, MAX_PAGES + 1):
        url = get_page_url(page)
        logging.info(f"🌐 Fetching page {page}: {url}")
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "html.parser")
            threads = extract_threads(soup, BASE_URL)
            if not threads:
                logging.info("⛔ No more threads found. Stopping.")
                break
            all_threads.extend(threads)
            logging.info(f"✅ Extracted {len(threads)} threads from page {page}")
            time.sleep(DELAY)
        except Exception as e:
            logging.error(f"❌ Request failed: {e}")
            break
    return pd.DataFrame(all_threads)

# Run scraper
logging.info("🚀 Starting scraper...")
df_raw = scrape_breast_cancer_forum()
df_raw.to_csv(f"{OUTPUT_DIR}/macmillan_bc_threads.csv", index=False)
print(f"✅ Saved {len(df_raw)} threads to Drive.")

✅ Saved 0 threads to Drive.


# **Cell 4: Topic & Pain Point Analysis**

In [ ]:
import pandas as pd
from collections import Counter

try:
    df = pd.read_csv(f"{OUTPUT_DIR}/macmillan_bc_threads.csv")
except pd.errors.EmptyDataError:
    print("⚠️ The CSV file is empty. No data to analyze. Please check the scraper in the previous step.")
    df = pd.DataFrame() # Create an empty DataFrame to avoid further errors

# Only proceed with analysis if the DataFrame is not empty
if not df.empty:
    # Breast cancer topic keywords
    TOPICS = {
        "Treatment Side Effects": ["chemo", "radiotherapy", "fatigue", "nausea", "hair loss", "neuropathy", "lymphoedema", "hot flush", "brain fog"],
        "Emotional Wellbeing": ["anxiety", "depression", "scared", "overwhelmed", "support", "counselling", "mental health", "coping", "terrified"],
        "Diagnosis & Waiting": ["waiting", "results", "biopsy", "mammogram", "delay", "nhs wait", "scan", "appointment"],
        "Financial & Work": ["work", "job", "money", "benefits", "sick leave", "insurance", "financial", "employer"],
        "Body Image & Identity": ["mastectomy", "reconstruction", "wig", "scar", "confidence", "intimacy", "body image"]
    }

    def classify(text):
        t = str(text).lower()
        return [topic for topic, kwds in TOPICS.items() if any(k in t for k in kwds)]

    df["assigned_topics"] = df["title"].apply(classify)

    # Extract pain point frequency
    pain_keywords = ["waiting", "delay", "confused", "no answer", "lost", "scared", "overwhelmed", "exhausted", "pain", "nausea"]
    df["pain_indicator"] = df["title"].str.lower().str.contains("|".join(pain_keywords), na=False)

    # Summary stats
    topic_counts = Counter(t for topics in df["assigned_topics"] for t in topics)
    pain_pct = round(df["pain_indicator"].mean() * 100, 1)

    print("📊 TOP 5 DISCUSSED TOPICS:")
    for t, c in topic_counts.most_common(5):
        print(f"  • {t}: {c} threads ({round(c/len(df)*100,1)}%)")
    print(f"\n⚠️ PAIN POINT INDICATOR: {pain_pct}% of threads contain high-distress keywords")
    print(f"📈 AVG REPLIES: {df['replies'].mean():.1f} | MEDIAN VIEWS: {df['views'].median():.0f}")

    # Save analysis summary
    summary_df = pd.DataFrame(topic_counts.most_common(), columns=["Topic", "Count"])
    summary_df.to_csv(f"{OUTPUT_DIR}/macmillan_bc_topic_summary.csv", index=False)
    print("✅ Summary saved to Drive.")
else:
    print("No data to generate analysis summary.")

EmptyDataError: No columns to parse from file

# **Cell 5: Infographic-Ready Visualization**

In [ ]:
# @title 🎨 Step 3: Generate Infographic Charts
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Topic Distribution
top5 = summary_df.head(5)
axes[0].barh(top5["Topic"], top5["Count"], color="#E88D9A", edgecolor="#C45A6C")
axes[0].set_title("🔝 Top 5 Discussion Topics", fontsize=14, fontweight="bold")
axes[0].invert_yaxis()

# Chart 2: Engagement vs Pain Points
df["pain_flag"] = df["pain_indicator"].map({True: "High Distress", False: "Standard"})
ax2 = sns.boxplot(data=df, x="pain_flag", y="replies", ax=axes[1], palette=["#4A90E2", "#E88D9A"])
axes[1].set_title("⚡ Engagement by Distress Level", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Reply Count")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/macmillan_bc_infographic_charts.png", dpi=300)
plt.show()
print("✅ Charts saved to Drive. Use PNG for your infographic layout.")